# Semana 2 — Superposición, Medición e Incertidumbre
## Cuando el qubit existe en dos mundos a la vez

**Objetivo:** Entender cómo la superposición cuántica difiere de la clásica, modelar la medición como colapso de estado y cuantificar la incertidumbre.

**Hito verificable:** Puedes simular 1000 mediciones de cualquier estado |ψ⟩ y verificar que las frecuencias convergen a |α|² y |β|².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('Librerías importadas correctamente')

## Ejercicio 2.1 — El estado |+⟩ y el estado |−⟩
Los estados de superposición más importantes: |+⟩ = (|0⟩+|1⟩)/√2  y  |−⟩ = (|0⟩−|1⟩)/√2

In [ ]:
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

ket_plus  = (ket_0 + ket_1) / np.sqrt(2)
ket_minus = (ket_0 - ket_1) / np.sqrt(2)

print(f'|+⟩ = {ket_plus}')
print(f'|−⟩ = {ket_minus}')
print(f'Norma |+⟩ = {np.linalg.norm(ket_plus):.10f}')
print(f'Norma |−⟩ = {np.linalg.norm(ket_minus):.10f}')
print()
print(f'⟨+|−⟩ = {np.dot(ket_plus.conj(), ket_minus):.10f}  → deben ser ortogonales')
print()
print('P(medir |0⟩) en |+⟩:', abs(ket_plus[0])**2)
print('P(medir |1⟩) en |+⟩:', abs(ket_plus[1])**2)
print('→ Resultado completamente aleatorio, 50/50')

## Ejercicio 2.2 — Simular el proceso de medición
La medición colapsa el estado cuántico. Podemos simularla con `numpy.random.choice`.

In [ ]:
def medir_estado(psi: np.ndarray, n_shots: int = 1000) -> dict:
    """Simula n_shots mediciones del estado psi en la base computacional."""
    probs = np.abs(psi)**2  # probabilidades: |α|² y |β|²
    resultados = np.random.choice(len(psi), size=n_shots, p=probs)
    conteos = {str(i): int(np.sum(resultados == i)) for i in range(len(psi))}
    return conteos

# Estado de superposición |+⟩
conteos = medir_estado(ket_plus, n_shots=10000)
print(f'10 000 mediciones de |+⟩:')
for estado, n in conteos.items():
    etiqueta = f'|{estado}⟩'
    barra = '█' * (n // 200)
    print(f'  {etiqueta}: {n:5d}  ({100*n/10000:.1f}%)  {barra}')

print()
print('Estado sesgado α=√(3)/2, β=1/2  →  P(0)=75%, P(1)=25%:')
psi_sesgado = np.array([np.sqrt(3)/2, 1/2], dtype=complex)
conteos2 = medir_estado(psi_sesgado, n_shots=10000)
for estado, n in conteos2.items():
    print(f'  |{estado}⟩: {n:5d}  ({100*n/10000:.1f}%)')

## Ejercicio 2.3 — Convergencia: ley de los grandes números cuántica
Con pocas mediciones hay mucho ruido; con muchas, la frecuencia converge a la probabilidad teórica.

In [ ]:
shots_list = [10, 50, 100, 500, 1000, 5000, 10000]
psi_test = np.array([np.sqrt(3)/2, 1/2], dtype=complex)
p_teorica_0 = abs(psi_test[0])**2  # 0.75

print(f'Estado: α=√3/2, β=1/2  →  P(|0⟩) teórica = {p_teorica_0:.4f}')
print()
print(f'{"shots":>8}  {"P(|0⟩) medida":>16}  {"error":>10}')
print('-' * 40)

for shots in shots_list:
    c = medir_estado(psi_test, n_shots=shots)
    p_medida = c['0'] / shots
    error = abs(p_medida - p_teorica_0)
    print(f'{shots:>8}  {p_medida:>16.4f}  {error:>10.4f}')

print()
print('→ El error disminuye como 1/√N (incertidumbre estadística clásica)')

## Ejercicio 2.4 — Principio de incertidumbre de Heisenberg
Hay pares de observables que no pueden medirse con precisión simultánea. Posición-momento, pero también Z-X en qubits.

In [ ]:
# Operadores de Pauli
Z = np.array([[1,  0], [0, -1]], dtype=complex)  # observable de la base computacional
X = np.array([[0,  1], [1,  0]], dtype=complex)  # observable de la base de Hadamard

def valor_esperado(psi, observable):
    """⟨ψ|O|ψ⟩"""
    return np.real(psi.conj() @ observable @ psi)

def varianza(psi, observable):
    """⟨O²⟩ - ⟨O⟩²"""
    ev = valor_esperado(psi, observable)
    ev2 = valor_esperado(psi, observable @ observable)
    return ev2 - ev**2

estados = [
    ('|0⟩',  np.array([1, 0], dtype=complex)),
    ('|1⟩',  np.array([0, 1], dtype=complex)),
    ('|+⟩',  np.array([1, 1], dtype=complex) / np.sqrt(2)),
    ('|−⟩',  np.array([1, -1], dtype=complex) / np.sqrt(2)),
]

print(f'{"Estado":>6}  {"⟨Z⟩":>8}  {"ΔZ":>8}  {"⟨X⟩":>8}  {"ΔX":>8}  {"ΔZ·ΔX":>10}')
print('-' * 60)
for nombre, psi in estados:
    ez = valor_esperado(psi, Z)
    ex = valor_esperado(psi, X)
    dz = np.sqrt(max(varianza(psi, Z), 0))
    dx = np.sqrt(max(varianza(psi, X), 0))
    print(f'{nombre:>6}  {ez:>8.4f}  {dz:>8.4f}  {ex:>8.4f}  {dx:>8.4f}  {dz*dx:>10.4f}')

print()
print('REGLA: Si el estado es autovector de Z → ΔZ=0 pero ΔX=1 (incertidumbre máxima en X)')
print('No se puede conocer Z y X simultáneamente con precisión arbitraria.')

## Ejercicio 2.5 — El colapso de la onda y mediciones repetidas
Una segunda medición sobre el mismo estado colapsado siempre da el mismo resultado.

In [ ]:
def colapsar_estado(psi: np.ndarray) -> tuple:
    """Simula una medición: devuelve (resultado, estado_colapsado)."""
    probs = np.abs(psi)**2
    resultado = np.random.choice(len(psi), p=probs)
    # El estado colapsa al autovector correspondiente
    estado_post = np.zeros(len(psi), dtype=complex)
    estado_post[resultado] = 1.0
    return resultado, estado_post

print('Experimento: medir |+⟩, luego medir el estado resultante 5 veces más')
print()
for experimento in range(5):
    r1, psi_colapsado = colapsar_estado(ket_plus)
    mediciones_siguientes = [colapsar_estado(psi_colapsado)[0] for _ in range(4)]
    print(f'  Experimento {experimento+1}: primera medición → |{r1}⟩, '
          f'siguientes: {mediciones_siguientes}')

print()
print('→ Tras el primer colapso, el estado es determinista: siempre da el mismo resultado.')
print('→ La aleatoriedad es intrínseca al acto de medir, no al estado pre-medición.')

## Mini reto — Completar antes de avanzar a la Semana 3

Implementa `estadistica_completa(psi, n_shots)` que devuelva:
- Media y desviación estándar de los resultados de medición
- Frecuencias medidas vs. probabilidades teóricas
- Un histograma usando matplotlib

Pruébalo con 5 estados distintos (incluyendo uno con amplitudes complejas).

In [ ]:
# TU CÓDIGO AQUÍ
def estadistica_completa(psi: np.ndarray, n_shots: int = 1000) -> None:
    """Muestra estadísticas completas de medición del estado psi."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 2

Antes de avanzar, comprueba que puedes:

- [ ] Construir |+⟩ y |−⟩ y verificar que son ortogonales
- [ ] Simular mediciones y obtener frecuencias cercanas a las probabilidades teóricas
- [ ] Explicar por qué se necesitan muchas mediciones para estimar probabilidades
- [ ] Calcular el valor esperado ⟨ψ|Z|ψ⟩
- [ ] Explicar qué es el colapso de la función de onda

## Reflexión (escribe aquí tu respuesta)

**¿Por qué la medición cuántica es fundamentalmente diferente a una medición clásica?**

*(tu respuesta aquí)*

**¿Qué significa que Z y X no conmutan físicamente?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 3 — Puertas Cuánticas de Un Qubit*